# Ebola weekly map tour — DRC

A JupyterGIS-ready spatiotemporal storymap for reviewing reported Ebola cases and deaths by week and locality across the complete HDX time span. This is an epidemiological data review, not a transmission-risk map.

## Tour scope and data discipline
The tour uses the complete HDX/INRB-UMIE consolidated CSV retained in this repository. It aggregates reported cumulative cases and deaths into epidemiological weeks, keeps reference dates visible, and separates reported observations from mobility and contextual layers.

## Sources and provenance
HDX source: https://data.humdata.org/dataset/republique-democratique-du-congo-cas-et-deces-d-ebola. The source CSV has locality names and dates but no coordinates, so the map uses documented approximate locality coordinates only for visualization. Validate each location and date against the original provider release.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, IFrame, display

ebola_url = 'https://raw.githubusercontent.com/jltobias/JupyterLite-DRC-Population-Mobility-Border-Mapping/main/data/ebola/drc_ebola_cases_consolidated.csv'
local_ebola_url = 'data/ebola/drc_ebola_cases_consolidated.csv'
try:
    ebola = pd.read_csv(local_ebola_url, parse_dates=['reference_date'])
    ebola_source = 'bundled JupyterLite contents copy'
except (FileNotFoundError, OSError):
    ebola = pd.read_csv(ebola_url, parse_dates=['reference_date'])
    ebola_source = 'HDX CSV via repository URL'
ebola = ebola[ebola['measure'].isin(['cases', 'deaths'])].copy()
ebola['value'] = pd.to_numeric(ebola['value'], errors='coerce').fillna(0)
ebola['week'] = ebola['reference_date'].dt.to_period('W-SUN').apply(lambda period: period.start_time)
ebola[['location_name', 'reference_date', 'week', 'measure', 'case_classification', 'value']].head()

In [ ]:
# Approximate locality coordinates used only to place the mapped subset.
coords = {
    'Bunia': (30.25, 1.56), 'Mongbalu': (30.02, 1.95), 'Rwampara': (30.31, 1.55),
    'Butembo': (29.29, 0.13), 'Goma': (29.23, -1.68), 'Katwa': (29.37, 0.13),
    'Nyakunde': (29.78, 1.26), 'Beni': (29.47, 0.49), 'Aru': (30.83, 2.86),
    'Mahagi': (30.99, 2.15), 'Mambasa': (29.18, 1.56), 'Isiro': (27.62, 2.77),
    'Makiso-Kisangani': (25.19, 0.52),
}
coords

## Weekly aggregation
The weekly tour is an exploratory summary of cumulative provider values. It is not a weekly incidence calculation: do not subtract or compare cumulative values without checking revisions, missing dates, and case definitions.

In [ ]:
weekly = (
    ebola.groupby(['week', 'measure'], as_index=False)['value'].sum()
         .pivot(index='week', columns='measure', values='value')
         .fillna(0)
         .sort_index()
)
weekly.tail()

In [ ]:
weekly.plot(figsize=(11, 4), color={'cases': '#d73027', 'deaths': '#542788'}, title='Reported cumulative Ebola values by epidemiological week')
plt.ylabel('Reported cumulative value')
plt.xlabel('Week beginning')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Place × time case surface
The next view highlights where the source reports values and when they change. Blank cells mean no row was present for that place/week; they should not automatically be treated as zero.

In [ ]:
place_week = (
    ebola[ebola['location_name'].isin(coords)]
    .groupby(['location_name', 'week', 'measure'], as_index=False)['value'].sum()
    .query("measure == 'cases'")
    .pivot(index='location_name', columns='week', values='value')
)
fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(place_week.fillna(0), aspect='auto', cmap='Reds')
ax.set_yticks(range(len(place_week.index)), place_week.index)
ax.set_xticks(range(len(place_week.columns)), [str(x.date()) for x in place_week.columns], rotation=60, ha='right')
ax.set_title('Reported cumulative cases by mapped locality and week')
ax.set_xlabel('Week beginning')
fig.colorbar(im, ax=ax, label='Reported cumulative cases')
plt.tight_layout()
plt.show()

In [ ]:
weekly_places = (
    ebola[ebola['location_name'].isin(coords)]
    .groupby(['week', 'location_name', 'measure'], as_index=False)['value'].sum()
)
weekly_places.head()

## Weekly place map
Use `plot_week(week_start)` to inspect one week at a time. Marker area is scaled by reported cumulative cases; marker color distinguishes cases from deaths. Approximate coordinates are disclosed in the source metadata.

In [ ]:
from matplotlib.lines import Line2D

def plot_week(week_start):
    selected = weekly_places[weekly_places['week'].eq(pd.Timestamp(week_start))]
    values = selected.pivot(index='location_name', columns='measure', values='value').fillna(0)
    fig, ax = plt.subplots(figsize=(9, 6))
    for name, (lon, lat) in coords.items():
        cases = float(values.loc[name, 'cases']) if name in values.index and 'cases' in values else 0
        deaths = float(values.loc[name, 'deaths']) if name in values.index and 'deaths' in values else 0
        ax.scatter(lon, lat, s=25 + cases ** 0.5 * 8, color='#d73027', alpha=0.72, edgecolor='white')
        if deaths > 0:
            ax.scatter(lon, lat, s=8 + deaths ** 0.5 * 5, color='#542788', alpha=0.75, edgecolor='white')
        ax.text(lon + 0.06, lat + 0.03, name, fontsize=7)
    ax.set(xlabel='Longitude', ylabel='Latitude', title=f'Reported Ebola values — week beginning {pd.Timestamp(week_start).date()}')
    ax.grid(alpha=0.2)
    ax.legend(handles=[Line2D([0], [0], marker='o', color='w', label='Cases', markerfacecolor='#d73027', markersize=8), Line2D([0], [0], marker='o', color='w', label='Deaths', markerfacecolor='#542788', markersize=8)])
    plt.tight_layout()
    plt.show()

plot_week(weekly_places['week'].min())

## Preview navigation: weekly place-and-time tour
The navigator pages through every available epidemiological week. Each page lists mapped locations, reported cases/deaths, the reference week, and a Google Street View link built from the approximate map coordinate. Street View coverage and imagery dates vary; open the link to inspect the ground context and do not treat imagery as case evidence.

In [ ]:
mapped = ebola[ebola['location_name'].isin(coords)].copy()
latest_by_week_place = mapped.groupby(['week', 'location_name', 'measure'], as_index=False)['value'].sum()
weeks = sorted(latest_by_week_place['week'].dropna().unique())
frames = []
for week in weeks:
    subset = latest_by_week_place[latest_by_week_place['week'].eq(week)]
    rows = []
    for name, (lon, lat) in coords.items():
        row = subset[subset['location_name'].eq(name)].set_index('measure')['value']
        rows.append({'name': name, 'cases': float(row.get('cases', 0)), 'deaths': float(row.get('deaths', 0)), 'streetview': f'https://www.google.com/maps/@?api=1&map_action=pano&viewpoint={lat},{lon}&heading=0&pitch=0&fov=90'})
    frames.append({'week': str(pd.Timestamp(week).date()), 'rows': rows})

payload = json.dumps(frames)
html = """<style>.tour{font:14px system-ui;max-width:900px;border:1px solid #ccd;padding:16px;border-radius:10px}.tour button{margin:3px;padding:7px 12px}.tour table{border-collapse:collapse;width:100%;margin-top:12px}.tour th,.tour td{border-bottom:1px solid #ddd;padding:5px;text-align:left}</style><div class='tour'><h3 id='tour-title'></h3><button id='prev'>Previous week</button><button id='next'>Next week</button><span id='count'></span><div id='tour-body'></div></div><script>const frames = """ + payload + """; let i=0; const esc=s=>String(s).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','\"':'&quot;',"'":'&#39;'}[c])); function render(){const f=frames[i]; document.getElementById('tour-title').textContent='Ebola map tour — week beginning '+f.week; document.getElementById('count').textContent='  '+(i+1)+' / '+frames.length; document.getElementById('tour-body').innerHTML='<table><thead><tr><th>Location</th><th>Cases</th><th>Deaths</th><th>Ground view</th></tr></thead><tbody>'+f.rows.map(r=>'<tr><td>'+esc(r.name)+'</td><td>'+r.cases+'</td><td>'+r.deaths+'</td><td><a target="_blank" rel="noopener" href="'+r.streetview+'">Open Street View</a></td></tr>').join('')+'</tbody></table>';} document.getElementById('prev').onclick=()=>{i=(i-1+frames.length)%frames.length;render()}; document.getElementById('next').onclick=()=>{i=(i+1)%frames.length;render()}; render();</script>"""
HTML(html)

## Case line listing
Use the line listing to audit the exact source rows behind the tour. Keep `reference_date`, `case_classification`, `time_period`, `source`, and `source_url` with any exported analysis.

In [ ]:
line_listing = ebola[['location_name', 'reference_date', 'measure', 'case_classification', 'time_period', 'value', 'source', 'source_url']].sort_values(['reference_date', 'location_name', 'measure'])
display(line_listing.head(50))

## Ground context links
Google Street View links are convenience links generated from approximate locality coordinates. They may open a nearby panorama, a map view, or no imagery. Respect Google terms and do not use street imagery to identify cases, patients, households, or sensitive sites.

In [ ]:
# Open the project's interactive layer map alongside this weekly tour.
IFrame('../../maps/maplibre-drc-mobility.html', width='100%', height=600)

## JupyterGIS handoff
Open this notebook in JupyterGIS to replace the schematic weekly map with a real GeoJSON layer or a time-enabled map. The repository contains the mapped HDX subset at data/ebola/drc-ebola-cases-deaths.geojson; its coordinates are approximate and its properties retain provider/date metadata.

In [ ]:
analysis_checklist = [
    'Confirm the provider release and reporting period.',
    'Check whether values are cumulative, revised, confirmed, or suspected.',
    'Do not interpret missing locality-week rows as zero.',
    'Do not infer transmission, individual movement, or risk from proximity to mobility flows.',
    'Retain approximate-coordinate and source limitations in every export.',
]
analysis_checklist